In [2]:
import pandas as pd
import numpy as np

In [4]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)

In [5]:
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [8]:
df.shape

(10000, 11)

# Question 1

In [10]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

# Question 2

In [11]:
df["horsepower"].median()

254.0

# Question 3

In [12]:
df = df.copy()

np.random.seed(42)

n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].copy()
df_val = df.iloc[idx[n_train:n_train + n_val]].copy()
df_test = df.iloc[idx[n_train + n_val:]].copy()

In [13]:
len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [14]:
features = [
    "engine_displacement",
    "horsepower",
    "vehicle_weight",
    "model_year"
]

target = "fuel_efficiency_mpg"

In [15]:
def train_linear_regression(X, y):
    XTX = X.T @ X
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv @ X.T @ y
    
    return w

In [16]:
def train_linear_regression(X, y):
    X = np.column_stack([np.ones(X.shape[0]), X])
    
    XTX = X.T @ X
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv @ X.T @ y
    
    return w

In [17]:
def predict(X, w):
    X = np.column_stack([np.ones(X.shape[0]), X])
    return X @ w

In [18]:
def rmse(y, y_pred):
    error = y - y_pred
    mse = np.mean(error ** 2)
    return np.sqrt(mse)

In [19]:
X_train = df_train[features].fillna(0).values
y_train = df_train[target].values

X_val = df_val[features].fillna(0).values
y_val = df_val[target].values

w = train_linear_regression(X_train, y_train)

y_pred = predict(X_val, w)

rmse(y_val, y_pred)

np.float64(2.2052931947534096)

In [20]:
mean_hp = df_train["horsepower"].mean()

mean_hp

np.float64(254.46338337605272)

In [21]:
X_train = df_train[features].fillna(mean_hp).values
X_val = df_val[features].fillna(mean_hp).values

w = train_linear_regression(X_train, y_train)

y_pred = predict(X_val, w)

rmse(y_val, y_pred)

np.float64(2.2018174845674756)

# Question 4

In [22]:
def train_linear_regression_reg(X, y, r):
    X = np.column_stack([np.ones(X.shape[0]), X])
    
    XTX = X.T @ X
    XTX = XTX + r * np.eye(XTX.shape[0])
    
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv @ X.T @ y
    
    return w

In [23]:
X_train = df_train[features].fillna(0).values
X_val = df_val[features].fillna(0).values

results = {}

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w = train_linear_regression_reg(X_train, y_train, r)
    y_pred = predict(X_val, w)
    
    score = rmse(y_val, y_pred)
    results[r] = round(score, 4)

results

{0: np.float64(2.2053),
 0.01: np.float64(2.2058),
 0.1: np.float64(2.2241),
 1: np.float64(2.3492),
 5: np.float64(2.4094),
 10: np.float64(2.4195),
 100: np.float64(2.4292)}

# Question 5

In [24]:
scores = []

for seed in range(10):
    np.random.seed(seed)
    
    idx = np.arange(n)
    np.random.shuffle(idx)
    
    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    
    X_train = df_train[features].fillna(0).values
    y_train = df_train[target].values
    
    X_val = df_val[features].fillna(0).values
    y_val = df_val[target].values
    
    w = train_linear_regression(X_train, y_train)
    
    y_pred = predict(X_val, w)
    
    score = rmse(y_val, y_pred)
    scores.append(score)

scores

[np.float64(2.2392041430461265),
 np.float64(2.202112821083884),
 np.float64(2.1634197787530947),
 np.float64(2.2043588768539215),
 np.float64(2.2018800943311554),
 np.float64(2.2457851509085134),
 np.float64(2.2697212079524043),
 np.float64(2.190794599933443),
 np.float64(2.216611201686444),
 np.float64(2.207036592817896)]

In [25]:
np.std(scores)

np.float64(0.02878127407819125)

# Question 6

In [26]:
np.random.seed(9)

idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]


In [27]:
df_full_train = pd.concat([df_train, df_val])

In [28]:
X_full_train = df_full_train[features].fillna(0).values
y_full_train = df_full_train[target].values

X_test = df_test[features].fillna(0).values
y_test = df_test[target].values

In [29]:
w = train_linear_regression_reg(
    X_full_train,
    y_full_train,
    r=0.001
)

In [30]:
y_pred = predict(X_test, w)

rmse(y_test, y_pred)

np.float64(2.235827747191731)